In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

# 환경 확인하기
env = gym.make('CartPole-v1')
print("State space:", env.observation_space.shape[0]) # 4
print("Action space:", env.action_space.n)            # 2

In [ ]:
print(env.observation_space)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x) # 마지막엔 활성화 함수 없음

In [ ]:
def get_action(state, epsilon, q_net, env):
    if random.random() < epsilon:
        return env.action_space.sample()
    else:
        # numpy array (4,) -> tensor (1, 4) 로 차원 확장
        state_tensor = torch.FloatTensor(state).unsqueeze(0) 
        with torch.no_grad():
            q_values = q_net(state_tensor)
            action = q_values.argmax().item()
        return action

In [ ]:
env = gym.make('CartPole-v1')
q_net = QNetwork(4, 2)
optimizer = optim.Adam(q_net.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

epsilon = 0.5
gamma = 0.99

for episode in range(100):
    state, _ = env.reset()
    total_reward = 0
    
    for t in range(200):
        action = get_action(state, epsilon, q_net, env)
        next_state, reward, done, truncated, _ = env.step(action)
        game_over = done or truncated
        
        # 예측값
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        predict_q = q_net(state_tensor)[0][action]
        
        # 타겟값
        next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0)
        with torch.no_grad():
            max_next_q = q_net(next_state_tensor).max().item()
            
        if game_over:
            target_q = torch.tensor(reward, dtype=torch.float32)
        else:
            target_q = torch.tensor(reward + gamma * max_next_q, dtype=torch.float32)
            
        # 역전파
        loss = loss_fn(predict_q, target_q)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        state = next_state
        total_reward += reward
        if game_over: break
            
    # 출력해보면 점수가 오르기는커녕 10~20점 대에서 진동하거나 망가지는 걸 볼 수 있음.
    print(f"Episode {episode}, Score: {total_reward}")